In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ==========================================
# 1. Parámetros de Amenaza y Sitio (NSR-10)
# Ubicación: Neiva, Huila (Parque Santander)
# Suelo: Perfil Tipo B (Roca)
# ==========================================
Aa = 0.25   # Aceleración pico efectiva
Av = 0.25   # Velocidad pico efectiva
Fa = 1.0    # Coeficiente de amplificación en periodo corto (Roca)
Fv = 1.0    # Coeficiente de amplificación en periodos intermedios (Roca)
I = 1.0     # Coeficiente de importancia (Grupo I)

# ==========================================
# 2. Cálculo de Periodos de Control (Segundos)
# ==========================================
T0 = 0.1 * (Av * Fv) / (Aa * Fa)   # Periodo inicial de la meseta (0.10 s)
TC = 0.48 * (Av * Fv) / (Aa * Fa)  # Periodo final de la meseta (0.48 s)
TL = 2.4 * Fv                      # Periodo de inicio zona de desp. constante (2.40 s)

# ==========================================
# 3. Vector de Periodos (0 a 4 segundos)
# ==========================================
# PEER recomienda un espaciamiento pequeño para mejor ajuste (ej. 0.01 s)
T = np.arange(0.0, 4.01, 0.01)
Sa = np.zeros_like(T)

# ==========================================
# 4. Cálculo del Espectro Elástico
# ==========================================
for i, t in enumerate(T):
    if t < T0:
        # Tramo inicial
        Sa[i] = 2.5 * Aa * Fa * I * (0.4 + 0.6 * (t / T0))
    elif t <= TC:
        # Meseta de aceleración máxima
        Sa[i] = 2.5 * Aa * Fa * I
    elif t <= TL:
        # Zona de velocidad constante
        Sa[i] = (1.2 * Av * Fv * I) / t
    else:
        # Zona de desplazamiento constante
        Sa[i] = (1.2 * Av * Fv * TL * I) / (t**2)

# ==========================================
# 5. Exportar a CSV para PEER
# ==========================================
# PEER pide un archivo con el Periodo en la columna 1 y el Sa en la columna 2
df = pd.DataFrame({'Period(s)': T, 'Sa(g)': Sa})
nombre_archivo = 'Espectro_Objetivo_Neiva_Roca.csv'
df.to_csv(nombre_archivo, index=False)
print(f"Archivo '{nombre_archivo}' generado con éxito para subir a PEER.")

# ==========================================
# 6. Graficar el Espectro
# ==========================================
plt.figure(figsize=(10, 6))
plt.plot(T, Sa, color='blue', linewidth=2.5, label='Espectro NSR-10')

# Rellenar la zona de la meseta para resaltar el máximo
T_meseta = T[(T >= T0) & (T <= TC)]
Sa_meseta = Sa[(T >= T0) & (T <= TC)]
plt.fill_between(T_meseta, Sa_meseta, color='blue', alpha=0.1)

plt.title('Espectro de Diseño Teórico NSR-10\nNeiva - Parque Santander (Perfil Tipo B)', fontsize=14, pad=15)
plt.xlabel('Periodo T (s)', fontsize=12)
plt.ylabel('Aceleración Espectral Sa (g)', fontsize=12)

# Configuración visual de la malla y los ejes
plt.grid(True, which='major', linestyle='-', alpha=0.6)
plt.grid(True, which='minor', linestyle=':', alpha=0.3)
plt.minorticks_on()
plt.xlim(0, 4)
plt.ylim(0, max(Sa) * 1.2)

# Anotaciones de los puntos clave
plt.axvline(x=T0, color='r', linestyle='--', alpha=0.5)
plt.axvline(x=TC, color='r', linestyle='--', alpha=0.5)
plt.axvline(x=TL, color='r', linestyle='--', alpha=0.5)
plt.text(T0 + 0.02, 0.1, f'T0={T0:.2f}s', rotation=90)
plt.text(TC + 0.02, 0.1, f'TC={TC:.2f}s', rotation=90)
plt.text(TL + 0.02, 0.1, f'TL={TL:.2f}s', rotation=90)

plt.legend()
plt.tight_layout()
plt.show()